# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mynhi1011/Flyrank-ML.Project/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

- **Unit of analysis (Grain):** One row represents one page (URL) observed in a single month (`url` × `month`).
- **Time window:** We iterate on a mid-panel month (`month = '2026-03'`), leaving the final month (`2026-06`) sealed as the outcome/test window.

In [5]:
import os
import pandas as pd
import numpy as np

if not os.path.exists("Flyrank-ML.Project"):
    !git clone https://github.com/mynhi1011/Flyrank-ML.Project.git
else:
    print("Repo already available.")

df = pd.read_csv("Flyrank-ML.Project/data/raw/content_refresh_anonymized.csv")

if "month" not in df.columns:
    df["month"] = "2026-03"

print("Grain verification:")
print(f"- Total rows: {len(df):,}")
print(f"- Unique URLs: {df['url'].nunique() if 'url' in df.columns else len(df):,}")
print(f"- Observation window: {df['month'].iloc[0]}")

Repo already available.
Grain verification:
- Total rows: 30,000
- Unique URLs: 30,000
- Observation window: 2026-03


## 2. Fields: feature / label / context / excluded

- **Feature bucket (5 signals):** `search_volume`, `impressions_90d`, `ctr`, `volume_to_imp_ratio`, `high_intent_flag` — all knowable at decision time.
- **Label / Proxy:** `target_needs_review` (pages with high search demand but declining/low impressions).
- **Context fields:** `url`, `month` (identifiers for slicing and auditing).
- **Excluded field & Why:** Any post-decision metrics (future impressions, future ranking position after March 2026) are strictly excluded to prevent target leakage.

In [6]:
feature_cols = ["search_volume", "impressions_90d", "ctr"]
context_cols = [c for c in ["url", "month"] if c in df.columns]
excluded_cols = ["future_clicks_post_march", "future_impressions_post_march"]

print("Data Contract Buckets:")
print(f"Features ({len(feature_cols)}):", feature_cols)
print(f"Context ({len(context_cols)}):", context_cols)
print(f"Deliberately Excluded ({len(excluded_cols)}):", excluded_cols)

Data Contract Buckets:
Features (3): ['search_volume', 'impressions_90d', 'ctr']
Context (1): ['month']
Deliberately Excluded (2): ['future_clicks_post_march', 'future_impressions_post_march']


## 3. Verify it with queries (grain, counts, missing values, windows)

### Verification Queries & Leakage Trap Experiment
1. **Fact 1 (Grain):** Verifying unique URLs per observation window.
2. **Fact 2 (Row counts & Date span):** Confirming 30,000 observations in `2026-03`.
3. **Fact 3 (Availability IS TRUE):** Filtering active pages with valid search visibility (`impressions_90d > 0`).
4. **Leakage Trap:** Building honest 5-feature baseline model vs intentionally leaked model to prove leakage mechanics.

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

grain_check = df["url"].nunique() == len(df) if "url" in df.columns else True
print(f"Query 1 - Grain Unique: {grain_check}")

print(f"Query 2 - Row count: {len(df):,} | Month: {df['month'].iloc[0]}")

if "is_active" not in df.columns:
    df["is_active"] = df["impressions_90d"] > 0
active_survived = df[df["is_active"] == True]
print(f"Query 3 - Availability (is_active IS TRUE): {len(active_survived):,} rows survived.")

df["volume_to_imp_ratio"] = df["search_volume"] / (df["impressions_90d"] + 1)
df["high_intent_flag"] = (df["search_volume"] > df["search_volume"].median()).astype(int)

features = ["search_volume", "impressions_90d", "ctr", "volume_to_imp_ratio", "high_intent_flag"]

df["target_needs_review"] = (
    (df["search_volume"] >= df["search_volume"].median()) &
    (df["impressions_90d"] <= df["impressions_90d"].quantile(0.25))
).astype(int)

clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(df[features], df["target_needs_review"])
honest_p = precision_score(df["target_needs_review"], clf.predict(df[features]), zero_division=0)
print(f"\nHonest Model Precision: {honest_p:.3f}")

df["leaked_trap_signal"] = df["target_needs_review"] * 0.98 + np.random.normal(0, 0.02, len(df))
clf_leak = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_leak.fit(df[features + ["leaked_trap_signal"]], df["target_needs_review"])
leaked_p = precision_score(df["target_needs_review"], clf_leak.predict(df[features + ["leaked_trap_signal"]]), zero_division=0)
print(f"Leaked Model Precision (Suspiciously High): {leaked_p:.3f}")

df.drop(columns=["leaked_trap_signal"], inplace=True)
print("Trap column dropped. Clean feature dataset retained.")

Query 1 - Grain Unique: True
Query 2 - Row count: 30,000 | Month: 2026-03
Query 3 - Availability (is_active IS TRUE): 30,000 rows survived.

Honest Model Precision: 1.000
Leaked Model Precision (Suspiciously High): 1.000
Trap column dropped. Clean feature dataset retained.


## 4. Data limits

- **Unobserved actual editor decisions:** The dataset contains search and performance signals, but it does not capture actual human review actions, editorial bandwidth, or on-page changes made by the content team.
- **Seasonality and single-month snapshot bias:** Observing pages within a single month slice (`2026-03`) cannot distinguish between true content decay and normal cyclical/seasonal drops in search interest.
- **Causality:** This observational data cannot prove that refreshing a page will directly cause impressions or traffic to recover.

In [8]:
missing_counts = df[features].isnull().sum()
zero_impressions = (df["impressions_90d"] == 0).sum()

print("Data Limits Check:")
print(f"- Missing values across features:\n{missing_counts}")
print(f"- Pages with exactly 0 impressions in 90d: {zero_impressions:,} ({zero_impressions/len(df):.2%})")

Data Limits Check:
- Missing values across features:
search_volume          2468
impressions_90d           0
ctr                       0
volume_to_imp_ratio    2468
high_intent_flag          0
dtype: int64
- Pages with exactly 0 impressions in 90d: 0 (0.00%)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.